In [10]:
# 0. imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# 1. load
df = pd.read_csv('student-por.csv')  # change path
df=df.drop(columns=["school","address","famsize","Pstatus","guardian","nursery","Dalc","Walc","reason"])

# 2. create CGPA target (10-scale) using simple average
df['avg_score'] = df[['G1','G2','G3']].mean(axis=1)
df['CGPA_10'] = (df['avg_score'] / 20.0) * 10.0
df['CGPA_10'] = df['CGPA_10'].round(2)

# 3. features and target
# keep G1 & G2 as features now
X = df.drop(['CGPA_10','avg_score'], axis=1)
y = df['CGPA_10']

# 4. split BEFORE preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. feature groups (include G1,G2 in numeric)
numeric_features = [
    'age','Medu','Fedu','traveltime','studytime',
    'failures','absences','famrel','freetime','goout','health',
    'G1','G2'   # <-- added
]

binary_features = ['sex','schoolsup','famsup','paid','activities','higher','internet','romantic']
multi_cat_features = ['Mjob','Fjob']

# 6. transformers (absences handling + binary encoder as before)
def transform_absences(X_df):
    X_df = X_df.copy()
    if 'absences' in X_df.columns:
        Q1 = X_df['absences'].quantile(0.25)
        Q3 = X_df['absences'].quantile(0.75)
        IQR = Q3 - Q1
        upper = Q3 + 1.5 * IQR
        lower = Q1 - 1.5 * IQR
        X_df['absences'] = np.clip(X_df['absences'], lower, upper)
        X_df['absences'] = np.log1p(X_df['absences'])
    return X_df

from sklearn.preprocessing import FunctionTransformer
absences_transformer = FunctionTransformer(transform_absences)

def binary_encoder(X_df):
    X_df = X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})
    # ensure numeric dtype for these cols (avoid FutureWarning)
    for c in X_df.columns:
        try:
            X_df[c] = pd.to_numeric(X_df[c])
        except:
            pass
    return X_df

binary_transformer = FunctionTransformer(binary_encoder)

numeric_pipeline = Pipeline([
    ('abs_fix', absences_transformer),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

binary_pipeline = Pipeline([('bin_enc', binary_transformer)])
cat_pipeline = Pipeline([('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('bin', binary_pipeline, binary_features),
    ('cat', cat_pipeline, multi_cat_features)
], remainder='drop')

# 7. full pipeline with model
full_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

# 8. fit
full_pipeline.fit(X_train, y_train)

# 9. predict & eval
y_pred = full_pipeline.predict(X_test)
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# 10. save pipeline (preprocessing + model together)
joblib.dump(full_pipeline, 'cgpa_pipeline_rf.pkl')
print("Saved pipeline to cgpa_pipeline_rf.pkl")


C:\Users\HP\AppData\Local\Temp\ipykernel_11484\1637310295.py:59: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_df = X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})


R2: 0.9741400949446186
MAE: 0.1512888461538467
RMSE: 0.23400834612827875
Saved pipeline to cgpa_pipeline_rf.pkl


C:\Users\HP\AppData\Local\Temp\ipykernel_11484\1637310295.py:59: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_df = X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})


In [12]:
X_train.shape

(519, 24)